In [ ]:
!pip install transformers sentencepiece langdetect nltk sentence-transformers scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 981.5/981.5 kB 9.2 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
  Created wheel for langdetect: filename=langdetect-1.0.9-py3-none-any.whl size=993223 sha256=701d63ccbc2cda29169e6b07252ea0c23ba6d6376b55958c6aeecaab7f038d5c
  Stored in directory: /root/.cache/pip/wheels/c1/67/88/e844b5b022812e15a52e4eaa38a1e709e99f06f6639d7e3ba7
Successfully built langdetect


In [ ]:
# Core libraries
from transformers import M2M100ForConditionalGeneration, M2M100Tokenizer
from langdetect import detect
from nltk.translate.bleu_score import sentence_bleu
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import time

In [ ]:
# Load translation model (supports 100+ languages)
model_name = "facebook/m2m100_418M"

tokenizer = M2M100Tokenizer.from_pretrained(model_name)
model = M2M100ForConditionalGeneration.from_pretrained(model_name)

# Load embedding model for semantic similarity
embedder = SentenceTransformer('all-MiniLM-L6-v2')

print("Model loaded successfully")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


tokenizer_config.json:   0%|          | 0.00/298 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/2.42M [00:00<?, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Model loaded successfully


In [ ]:
def translate(text, src_lang="en", tgt_lang="fr"):
    """
    Translate a single sentence
    """
    tokenizer.src_lang = src_lang

    encoded = tokenizer(text, return_tensors="pt")

    generated_tokens = model.generate(
        **encoded,
        forced_bos_token_id=tokenizer.get_lang_id(tgt_lang)
    )

    return tokenizer.batch_decode(generated_tokens, skip_special_tokens=True)[0]

In [ ]:
def translate_paragraph(text, src="en", tgt="fr"):
    """
    Translate paragraph by splitting into sentences
    """
    sentences = text.split(".")
    translated = []

    for sentence in sentences:
        if sentence.strip():
            translated.append(translate(sentence.strip(), src, tgt))

    return ". ".join(translated)

In [ ]:
def is_complex(text):
    """
    Detect whether text is complex using semantic similarity
    """
    sentences = text.split(".")

    if len(sentences) < 2:
        return False

    embeddings = embedder.encode(sentences)

    similarities = []
    for i in range(len(embeddings) - 1):
        sim = cosine_similarity([embeddings[i]], [embeddings[i+1]])[0][0]
        similarities.append(sim)

    avg_similarity = np.mean(similarities)

    # Lower similarity → multiple contexts → complex
    return avg_similarity < 0.5

In [ ]:
def smart_translate(text, target_lang="fr"):
    """
    Automatically:
    - Detect language
    - Decide sentence vs paragraph
    """
    detected_lang = detect(text)
    print("Detected Language:", detected_lang)

    # Check if input is English
    if detected_lang != "en":
        print("[WARNING] This system is optimized for English input. Results may vary.")

    # Decision logic
    if is_complex(text) or len(text.split()) > 15 or "." in text:
        print("[DECISION] Mode: Paragraph Translation")
        print("[INFO] Using function: translate_paragraph()")
        result = translate_paragraph(text, detected_lang, target_lang)
    else:
        print("[DECISION] Mode: Sentence Translation")
        print("[INFO] Using function: translate()")
        result = translate(text, detected_lang, target_lang)

    return result

In [ ]:
def evaluate_bleu(reference, translated):
    """
    BLEU score for lexical similarity
    """
    ref = [reference.split()]
    cand = translated.split()

    return sentence_bleu(ref, cand)


def evaluate_embedding(reference, translated):
    """
    Semantic similarity using embeddings
    """
    emb1 = embedder.encode(reference)
    emb2 = embedder.encode(translated)

    sim = cosine_similarity([emb1], [emb2])
    return sim[0][0]

In [ ]:
def measure_speed(text, target_lang):
    """
    Measures translation time using smart_translate
    """
    start = time.time()
    result = smart_translate(text, target_lang)
    end = time.time()

    return result, round(end - start, 3)

In [ ]:
# Supported language codes:
# en = English, fr = French, de = German, hi = Hindi,
# es = Spanish, zh = Chinese, ar = Arabic
valid_langs = ["en", "fr", "de", "hi", "es", "zh", "ar"]

while True:
    print("\n===== AI Translator Menu =====")
    print("1. Translate Text")
    print("2. Exit")

    choice = input("Enter choice: ")

    if choice == "2":
        print("Exiting system")
        break

    elif choice == "1":
        text = input("\nEnter text: ")
        target_lang = input("Enter target language (fr/de/hi/es/en): ")

        if target_lang not in valid_langs:
            print("Invalid language code")
            continue

        try:
            # Use reusable timing function
            translated, exec_time = measure_speed(text, target_lang)

            print("\nTranslated Output:", translated)
            print("Translation Time:", exec_time, "seconds")

            # Evaluation choice
            eval_choice = input("\nDo you want evaluation? (yes/no): ").lower()

            if eval_choice == "yes":
                ref = input("Enter reference translation: ")

                if ref.strip():
                    print("\nChoose evaluation method:")
                    print("1. BLEU Score")
                    print("2. Embedding Similarity")
                    print("3. Both")

                    eval_type = input("Enter choice: ")

                    if eval_type == "1":
                        bleu = evaluate_bleu(ref, translated)
                        print("BLEU Score:", round(bleu, 4))

                    elif eval_type == "2":
                        emb = evaluate_embedding(ref, translated)
                        print("Embedding Similarity:", round(emb, 4))

                    elif eval_type == "3":
                        bleu = evaluate_bleu(ref, translated)
                        emb = evaluate_embedding(ref, translated)

                        print("BLEU Score:", round(bleu, 4))
                        print("Embedding Similarity:", round(emb, 4))

                    else:
                        print("Invalid evaluation choice")

        except Exception as e:
            print("Error:", e)


===== AI Translator Menu =====
1. Translate Text
2. Exit
Enter choice: 1

Enter text: The rapid advancement of technology has transformed the way people communicate, work, and interact, creating both opportunities and challenges in modern society.
Enter target language (fr/de/hi/es/en): de
Detected Language: en
[DECISION] Mode: Paragraph Translation
[INFO] Using function: translate_paragraph()

Translated Output: Der schnelle Fortschritt der Technologie hat die Art und Weise verändert, wie Menschen kommunizieren, arbeiten und interagieren, was sowohl Möglichkeiten als auch Herausforderungen in der modernen Gesellschaft schafft.
Translation Time: 15.548 seconds

Do you want evaluation? (yes/no): yes
Enter reference translation: Der rasante Fortschritt der Technologie hat die Art und Weise verändert, wie Menschen kommunizieren, arbeiten und interagieren, und schafft sowohl Chancen als auch Herausforderungen in der modernen Gesellschaft.

Choose evaluation method:
1. BLEU Score
2. Embedd